[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB-X2_masked_autoencoder.ipynb)

# NB-X2 · Challenge: self-supervised pre-training with a masked autoencoder
**Day 5 · optional · about 45 minutes · GPU recommended.** Idea of the lecture: learn from **unlabelled** images by hiding patches and reconstructing them, then test with a **linear probe** whether the features are useful with few labels. This is a toy (28 px) version, not a replica of a published MAE.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
def set_seed(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s); torch.backends.cudnn.deterministic = True
set_seed(0); device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(device)
os.makedirs("data", exist_ok=True)
import medmnist; from medmnist import PneumoniaMNIST
tr = PneumoniaMNIST(split="train", download=True, root="data"); te = PneumoniaMNIST(split="test", download=True, root="data")
Xtr = torch.from_numpy(tr.imgs.astype(np.float32) / 255).unsqueeze(1); ytr = tr.labels.ravel(); Xte = torch.from_numpy(te.imgs.astype(np.float32) / 255).unsqueeze(1); yte = te.labels.ravel()
rng = np.random.default_rng(0); few = np.concatenate([rng.choice(np.where(ytr == c)[0], 25, replace=False) for c in (0, 1)])   # 50 labelled images
print("unlabelled pool:", len(Xtr), "| labelled subset:", len(few))

## 1 · Masking
Split the 28×28 image into 4×4 patches (49 patches). **TODO 1:** `random_patch_mask(x, ratio)` returns a mask of shape (B,1,28,28) with 1 where a patch is **hidden**. Hint: draw a (B,49) random matrix, hide the `ratio` fraction with the largest values, reshape to (B,1,7,7) and upsample with `repeat_interleave`.

In [ ]:
def random_patch_mask(x, ratio=0.6, patch=4):
    B, _, H, W = x.shape; g = H // patch; n = g * g; k = int(round(ratio * n))
    # TODO 1: random (B, n) values -> hide the k patches with ... -> reshape to (B,1,g,g) -> repeat_interleave(patch) on dims 2 and 3
    return torch.zeros(B, 1, H, W, device=x.device)

In [ ]:
set_seed(0); m = random_patch_mask(Xtr[:8], 0.6)
assert m.shape == (8, 1, 28, 28) and set(m.unique().tolist()) <= {0.0, 1.0}
assert abs(m.mean().item() - 0.6) < 0.05, "about 60% of the pixels should be hidden"
assert (m.view(8, 7, 4, 7, 4).std(dim=(2, 4)) == 0).all(), "masking must hide whole 4x4 patches"
print("Checkpoint 1 passed")

## 2 · Encoder + decoder, trained to reconstruct the hidden pixels

In [ ]:
class Enc(nn.Module):
    def __init__(s): super().__init__(); s.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU())
    def forward(s, x): return s.net(x)
class Dec(nn.Module):
    def __init__(s): super().__init__(); s.net = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(), nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.ReLU(), nn.Conv2d(16, 1, 3, padding=1))
    def forward(s, z): return s.net(z)
def pretrain(epochs=8, ratio=0.6, seed=0):
    set_seed(seed); enc, dec = Enc().to(device), Dec().to(device); opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), 1e-3)
    dl = DataLoader(TensorDataset(Xtr), batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(seed)); hist = []
    for ep in range(epochs):
        tot = 0
        for (xb,) in dl:
            xb = xb.to(device); m = random_patch_mask(xb, ratio); out = dec(enc(xb * (1 - m)))
            loss = (((out - xb) ** 2) * m).sum() / m.sum()          # loss ONLY on hidden pixels
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(xb)
        hist.append(tot / len(Xtr))
    return enc, dec, hist
enc, dec, hist = pretrain(); print("loss per epoch:", [round(h, 4) for h in hist])
assert hist[-1] < hist[0], "reconstruction loss should go down"

In [ ]:
enc.eval(); dec.eval(); set_seed(1); xb = Xte[:6].to(device); m = random_patch_mask(xb, 0.6)
with torch.no_grad(): rec = dec(enc(xb * (1 - m))).clamp(0, 1)
fig, ax = plt.subplots(3, 6, figsize=(9, 4.6))
for k in range(6):
    for r, (im, t) in enumerate([(xb * (1 - m), "input"), (rec, "reconstruction"), (xb, "original")]):
        ax[r, k].imshow(im[k, 0].cpu(), cmap="gray", vmin=0, vmax=1); ax[r, k].axis("off")
        if k == 0: ax[r, k].set_title(t, fontsize=8, loc="left")
plt.show()

## 3 · Linear probe with 50 labels
Freeze the encoder, average its features, train a logistic regression on the 50 labelled images, compare with the **same architecture, random weights, frozen** (the fair baseline). **TODO 2:** `features(enc, X)` = global average pooling of `enc(X)`.

In [ ]:
@torch.no_grad()
def features(enc, X, bs=512):
    enc.eval()
    # TODO 2: for each batch, enc(batch) -> mean over the spatial dims (2, 3) -> collect -> numpy array (N, 64)
    return np.zeros((len(X), 64))
def probe_auc(enc):
    clf = LogisticRegression(max_iter=3000).fit(features(enc, Xtr[few]), ytr[few]); return roc_auc_score(yte, clf.predict_proba(features(enc, Xte))[:, 1])
set_seed(0); rand_enc = Enc().to(device)
auc_ssl, auc_rand = probe_auc(enc), probe_auc(rand_enc)
print(f"linear probe, 50 labels: pre-trained encoder AUC {auc_ssl:.3f} | random frozen encoder AUC {auc_rand:.3f}")

In [ ]:
assert features(enc, Xte[:10]).shape == (10, 64)
print("Checkpoint 2 passed. The result is NOT guaranteed to favour pre-training at this scale: report what you see, with one seed this is an anecdote.")

**Reflection.** 1) Why is the loss computed only on hidden pixels? 2) Why compare with a *random frozen* encoder and not with nothing? 3) You used one seed and 50 labels: how would you turn this into a claim (repeat with 5 seeds, report the interval)? 4) Why might self-supervision help more in medical imaging than in natural images? (Many unlabelled scans, few expert labels.)